# Analisando a efetividade de medicamentos em medicina personalizada com algoritmos genéticos

Este projeto demonstra como implementar um algoritmo genético em Python para otimizar a  atribuição de medicamentos a pacientes, maximizando a efetividade total do tratamento   com  base nas características individuais dos pacientes e nos perfis dos medicamentos.  A  medicina personalizada tornando-se realidade através do aprendizado de máquina.

### Pacotes

In [2]:
!pip install -q deap

https://deap.readthedocs.io/en/master/

In [3]:
# Imports
import random
import numpy as np
from deap import base, creator, tools, algorithms

## Definindo pacientes, características e medicamentos

In [4]:
# Definir número de pacientes, características e medicamentos
N_PATIENTS = 50
N_FEATURES = 10
N_MEDICATIONS = 10

In [5]:
# Gerar características aleatórias para os pacientes (entre 0 e 1)
patient_features = np.random.rand(N_PATIENTS, N_FEATURES)

In [8]:
# Gerar perfis de efetividade aleatórios para os medicamentos
medication_profiles = np.random.rand(N_MEDICATIONS, N_FEATURES)

In [9]:
# Função para calcular a efetividade de um medicamento em um paciente
def calcula_efetividade(patient, medication):
    
    # A efetividade é calculada como o produto escalar entre as características do paciente 
    # e o perfil do medicamento
    return np.dot(patient, medication)

## Configuração do DEAP para o Algoritmo Genético

DEAP (Distributed Evolutionary Algorithms in Python): utilizado para configurar e executar um algoritmo genético. 

In [10]:
# Classe para avaliar a qualidade de uma solução
creator.create("FitnessMax", base.Fitness, weights = (1.0,)) 

`creator.create`: A função create do módulo creator da DEAP é usada para criar novas classes. Neste caso, estamos criando uma classe chamada "FitnessMax", que será derivada da classe base.Fitness.

`base.Fitness`: A classe Fitness é fornecida pela DEAP para avaliar a qualidade de uma solução (ou "indivíduo") em um algoritmo genético. base.Fitness é a classe base que é personalizada com o argumento weights.

`weights = (1.0,)`: O argumento weights indica o tipo de otimização que será feito na função de fitness. Aqui, (1.0,) especifica que o algoritmo deve maximizar a função de fitness (ou seja, buscamos o maior valor possível de efetividade). Um valor positivo (1.0) indica maximização, enquanto um valor negativo indicaria minimização. Esse valor é importante para orientar o algoritmo na direção correta.

In [11]:
# Cada Individual terá um atributo fitness
creator.create("Individual", list, fitness = creator.FitnessMax)

**creator.create**: Novamente, a função create é usada para criar uma nova classe chamada "Individual".

**list**: A classe Individual será derivada de uma lista Python nativa. Isso significa que cada "Indivíduo" será essencialmente uma lista, onde cada elemento representa uma característica ou gene. No contexto do projeto, cada item da lista pode representar a atribuição de um medicamento específico para um paciente.

**fitness = creator.FitnessMax**: Este atributo define que cada Individual terá um atributo fitness que usa a classe FitnessMax. Esse atributo é essencial para avaliar o quão boa é a solução representada pelo indivíduo.

As duas linhas acima configuram a estrutura básica para os indivíduos e a função de fitness no algoritmo genético. A configuração estabelece que:

- Cada indivíduo é uma lista de atributos ou "genes", que representa uma possível solução para o problema de atribuição de medicamentos.
- O objetivo é maximizar a função de fitness, que neste caso representa a eficácia total do tratamento para os pacientes, levando em consideração as características individuais e os perfis dos medicamentos.

Essa configuração é a base para que o algoritmo evolua em direção à melhor solução de maneira iterativa, ao combinar e mutar indivíduos ao longo de várias gerações.

In [12]:
toolbox = base.Toolbox()

In [13]:
# Gerador de atributos: atribui um medicamento aleatório a cada paciente
toolbox.register("attr_medication", random.randint, 0, N_MEDICATIONS - 1)

In [14]:
# Inicializadores de estrutura
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_medication, n = N_PATIENTS)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

## Função para avaliar a efetividade

A função calcula a eficácia total de uma solução específica, que é representada por uma lista de medicamentos atribuídos a pacientes. A eficácia de cada atribuição é somada para obter o total, que é usado como critério de fitness pelo algoritmo genético para avaliar o quão boa é a solução em maximizar a efetividade do tratamento.


In [15]:
# Função de avaliação que calcula a efetividade total de uma atribuição de medicamentos
def avalia_efetividade(individual):
    
    # Inicializa a variável para armazenar a efetividade total
    total_effectiveness = 0

    # Itera sobre cada paciente e a atribuição de medicamento correspondente no indivíduo
    for patient_idx, medication_idx in enumerate(individual):
        
        # Seleciona o perfil do paciente com base no índice atual
        patient = patient_features[patient_idx]
        
        # Seleciona o perfil do medicamento com base no índice atribuído
        medication = medication_profiles[medication_idx]
        
        # Calcula a efetividade do medicamento para o paciente usando uma função específica
        effectiveness = calcula_efetividade(patient, medication)
        
        # Adiciona a efetividade calculada ao total de efetividade
        total_effectiveness += effectiveness

    # Retorna a efetividade total como uma tupla (exigido pelo DEAP para o fitness)
    return (total_effectiveness,)

## Define os Operadores Genéticos

Essas linhas configuram os operadores genéticos essenciais para o algoritmo genético utilizando o toolbox do framework DEAP. O toolbox é uma ferramenta central no DEAP que registra funções que definem a lógica do algoritmo genético, como avaliação de fitness, crossover, mutação e seleção. 

In [16]:
# Registro da função de avaliação de fitness
toolbox.register("evaluate", avalia_efetividade)

**toolbox.register("evaluate", avalia_efetividade)**: Aqui, a função de avaliação avalia_efetividade é registrada com o nome "evaluate". Esta função será chamada para calcular a efetividade total (fitness) de cada indivíduo na população.

**Função de Fitness**: avalia_efetividade avalia o quão boa é uma solução (ou indivíduo) em maximizar a efetividade do tratamento com medicamentos. No contexto do DEAP, cada vez que o algoritmo precisar avaliar uma solução, ele chamará a função registrada como "evaluate".

In [17]:
# Registro do operador de crossover (cruzamento) uniforme
toolbox.register("mate", tools.cxUniform, indpb = 0.2)

**toolbox.register("mate", tools.cxUniform, indpb=0.2)**: Aqui, a função tools.cxUniform é registrada como o operador de crossover, com o nome "mate". O cxUniform é um tipo de crossover uniforme que decide aleatoriamente para cada gene se ele será trocado entre dois indivíduos (pais).

**indpb=0.2**: Esse parâmetro define a probabilidade individual (indpb) de cada gene ser trocado entre os pais. Com indpb=0.2, cada gene tem 20% de chance de ser trocado. Crossover uniforme é útil para criar maior variação nos filhos ao combinar características dos pais.

In [18]:
# Registro do operador de mutação
toolbox.register("mutate", tools.mutUniformInt, low = 0, up = N_MEDICATIONS - 1, indpb = 0.05)

**toolbox.register("mutate", tools.mutUniformInt, low=0, up=N_MEDICATIONS - 1, indpb=0.05)**: Essa linha registra tools.mutUniformInt como o operador de mutação com o nome "mutate". O mutUniformInt é um operador de mutação que altera o valor de genes específicos com base em um intervalo definido.

**low=0 e up=N_MEDICATIONS - 1**: Esses parâmetros especificam que o valor de cada gene mutado será alterado para um valor aleatório entre 0 e N_MEDICATIONS - 1, onde N_MEDICATIONS representa o número de medicamentos disponíveis. Portanto, cada gene (representando a atribuição de um medicamento) pode ser modificado dentro desse intervalo.

**indpb=0.05**: Esta é a probabilidade de mutação para cada gene. Com indpb=0.05, cada gene tem 5% de chance de sofrer mutação. A mutação ajuda a introduzir variação e evita que o algoritmo fique preso em soluções subótimas.

In [ ]:
# Registro do operador de seleção
toolbox.register("select", tools.selTournament, tournsize = 3)

**toolbox.register("select", tools.selTournament, tournsize=3)**: Aqui, o operador de seleção tools.selTournament é registrado com o nome "select". O selTournament é um método de seleção por torneio, onde grupos de indivíduos são selecionados aleatoriamente, e o mais apto dentro de cada grupo é escolhido para reprodução.

**tournsize=3**: O parâmetro tournsize define o tamanho do torneio, ou seja, quantos indivíduos competem entre si. Neste caso, três indivíduos são comparados a cada vez, e o que possui maior fitness é selecionado. A seleção por torneio é útil para balancear exploração e exploração, mantendo diversidade na população enquanto seleciona indivíduos mais aptos.